# 10 · Figures and Tables

**Reads the stored outputs of notebooks 03, 06, 07 and 09. Computes nothing.**

Regenerates every figure and table in the paper from result files, so re-running
an experiment propagates automatically and nothing is transcribed by hand. This
is what makes "zero unexplained mismatches between paper and notebook" achievable
rather than aspirational.

## Tables
1. **Label validation** — the four checks, with each marked confirmed or flagged.
2. **Control comparison** — the paper's central claim: same pipeline, different
   labels.
3. **Ablation** — every configuration, with interpretation flags preserved so a
   circular result cannot be read as a genuine improvement.

## Figures
1. **Validation** — kappa against the interpretive bands, the negative control,
   and instability under augmentation.
2. **Control comparison** — the gap between derived and expert labels from pixels
   alone.
3. **The gate** — per-fold gamma with its sign flips, the architectural evidence.
4. **Class distribution** — including why mild is unmeasurable at ~3 per fold.

## Missing sources are skipped, not faked
Anything absent is recorded in `figure_manifest.json` and its figure omitted. A
missing figure is honest; an invented one is not.

## Outputs
`figures/*.png` and `.pdf`, `tables/*.csv`, `figure_manifest.json`


In [1]:
# Cell 1 · config and inputs
from pathlib import Path
import pandas as pd, numpy as np, json, warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

INTERIM  = Path('/Users/mustaphanasser/Documents/computer_application/DFU_project/notebook2_outputs/interim')
OUT      = Path('/Users/mustaphanasser/Documents/computer_application/DFU_project/notebook2_outputs'); OUT.mkdir(exist_ok=True)
FIGS    = OUT / 'figures'; FIGS.mkdir(parents=True, exist_ok=True)
TABLES  = OUT / 'tables';  TABLES.mkdir(parents=True, exist_ok=True)

SOURCES = {
    'severity'  : OUT / 'results_severity.json',
    'infection' : OUT / 'results_infection.json',
    'stats'     : OUT / 'stats_report.json',
    'validation': INTERIM / 'validation_report.json',
    'labels'    : INTERIM / 'labels_consolidated.csv',
}
data, missing = {}, []
for k, p in SOURCES.items():
    if p.exists():
        data[k] = (pd.read_csv(p) if p.suffix == '.csv' else json.load(open(p)))
        print(f'  loaded {p.name}')
    else:
        missing.append(k); print(f'  MISSING {p.name}')

if not data:
    print('\nSTOPPING. Run notebooks 03-09 first.')
    raise SystemExit(1)

# house style, applied once
INK, MUTE, TEAL, AMBER, CRIM, GREEN = ('#14313D','#5B7683','#2E7D8F',
                                       '#D97706','#BE123C','#15803D')
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,
                     'axes.spines.top':False,'axes.spines.right':False,
                     'figure.dpi':110})
DPI = 300

def save(fig, name):
    for ext in ('png','pdf'):
        fig.savefig(FIGS / f'{name}.{ext}', dpi=DPI, bbox_inches='tight',
                    facecolor='white')
    plt.close(fig)
    print(f'  wrote figures/{name}.png and .pdf')

  loaded results_severity.json
  loaded results_infection.json
  loaded stats_report.json
  loaded validation_report.json
  loaded labels_consolidated.csv


In [2]:
# Cell 2 · Table 1 — label validation
# Every value read from validation_report.json. Nothing typed by hand, so
# a re-run of notebook 03 propagates here automatically.
rows = []
if 'validation' in data:
    v = data['validation']
    def add(check, measure, value, status):
        rows.append(dict(check=check, measure=measure, value=value,
                         status=status))
    c = v.get('check1_expert', {})
    if c.get('status') == 'confirmed':
        add('Expert agreement','derived vs Expert A',
            f"{c.get('kappa_derived_vs_A', float('nan')):+.4f}", 'confirmed')
        add('Expert agreement','derived vs Expert B',
            f"{c.get('kappa_derived_vs_B', float('nan')):+.4f}", 'confirmed')
        add('Expert agreement','Expert A vs B (inter-rater)',
            f"{c.get('kappa_inter_rater', float('nan')):+.4f}", 'confirmed')
    else:
        add('Expert agreement','Cohen kappa','not reproduced', 'FLAGGED')
    c = v.get('check2_negctrl', {})
    if c.get('status') == 'confirmed':
        add('Negative control','healthy skin graded severe',
            f"{c.get('pct_severe', float('nan')):.1f}%", 'confirmed')
        add('Negative control','max necrosis on intact skin',
            f"{c.get('max_necrosis', float('nan')):.3f}", 'confirmed')
    else:
        add('Negative control','healthy skin graded severe','not reproduced','FLAGGED')
    c = v.get('check3_stability', {})
    if c: add('Stability','photographs with >1 grade',
              f"{c.get('pct_inconsistent', float('nan')):.1f}%", c.get('status','?'))
    c = v.get('check4_plausibility', {})
    if c:
        add('Plausibility','mean necrotic fraction',
            f"{c.get('mean_necrosis', float('nan')):.3f}", c.get('status','?'))
        add('Plausibility','images above severe threshold',
            f"{c.get('pct_above_threshold', float('nan')):.1f}%", c.get('status','?'))

t1 = pd.DataFrame(rows)
if len(t1):
    t1.to_csv(TABLES / 'table1_validation.csv', index=False)
    print(t1.to_string(index=False))
    print(f'\n  wrote tables/table1_validation.csv')
else:
    print('  FLAGGED: no validation data')

           check                       measure   value    status
Expert agreement           derived vs Expert A -0.0254 confirmed
Expert agreement           derived vs Expert B +0.0359 confirmed
Expert agreement   Expert A vs B (inter-rater) +0.2537 confirmed
Negative control    healthy skin graded severe   60.2% confirmed
Negative control   max necrosis on intact skin   0.570 confirmed
       Stability     photographs with >1 grade   61.7% confirmed
    Plausibility        mean necrotic fraction   0.231 confirmed
    Plausibility images above severe threshold   63.1% confirmed

  wrote tables/table1_validation.csv


In [3]:
# Cell 3 · Table 2 — the control comparison
# The paper's central claim in one table: same pipeline, different labels.
rows = []
if 'severity' in data:
    cnn = [c for c in data['severity']['configs'] if c['name'].startswith('CNN only')]
    if cnn:
        b = max(cnn, key=lambda c: c['qwk'])
        rows.append(dict(task='Severity (derived colour labels)',
                         labels='k-means threshold on necrosis',
                         metric='QWK', value=f"{b['qwk']:.4f}",
                         best_config=b['name']))
        rows.append(dict(task='Severity (derived colour labels)',
                         labels='k-means threshold on necrosis',
                         metric='severe-vs-rest F1',
                         value=f"{b.get('severe_vs_rest_f1', float('nan')):.4f}",
                         best_config=b['name']))
if 'infection' in data:
    b = max(data['infection']['configs'], key=lambda c: c['auroc'])
    for m, k in [('AUROC','auroc'), ('AUPRC','auprc'), ('MCC','mcc')]:
        rows.append(dict(task='Infection (expert labels)',
                         labels='clinician-assigned',
                         metric=m, value=f"{b[k]:.4f}", best_config=b['name']))

t2 = pd.DataFrame(rows)
if len(t2):
    t2.to_csv(TABLES / 'table2_control.csv', index=False)
    print(t2.to_string(index=False))
    print('\n  Same architectures, preprocessing and protocol. Only the')
    print('  label source differs between the two blocks.')
    print(f'  wrote tables/table2_control.csv')
else:
    print('  FLAGGED: need severity and/or infection results')

                            task                        labels            metric  value        best_config
Severity (derived colour labels) k-means threshold on necrosis               QWK 0.2488 CNN only (softmax)
Severity (derived colour labels) k-means threshold on necrosis severe-vs-rest F1 0.7077 CNN only (softmax)
       Infection (expert labels)            clinician-assigned             AUROC 0.8101           CNN only
       Infection (expert labels)            clinician-assigned             AUPRC 0.8685           CNN only
       Infection (expert labels)            clinician-assigned               MCC 0.4637           CNN only

  Same architectures, preprocessing and protocol. Only the
  label source differs between the two blocks.
  wrote tables/table2_control.csv


In [4]:
# Cell 4 · Table 3 — ablation, with flags preserved
rows = []
for src, key, metric in [('severity','severity','qwk'),
                         ('infection','infection','auroc')]:
    if src not in data: continue
    for c in data[src]['configs']:
        rows.append(dict(task=key, configuration=c['name'],
                         metric=metric.upper(), value=f"{c[metric]:.4f}",
                         flag=c.get('interpretation_flag','')))
t3 = pd.DataFrame(rows)
if len(t3):
    t3.to_csv(TABLES / 'table3_ablation.csv', index=False)
    with pd.option_context('display.max_colwidth', 46):
        print(t3.to_string(index=False))
    print(f'\n  wrote tables/table3_ablation.csv')
    flagged = t3[t3.flag != '']
    if len(flagged):
        print(f'\n  {len(flagged)} row(s) carry an interpretation flag.')
        print('  Those must not be read as "tissue features improve results".')

     task             configuration metric  value                                                                                                                       flag
 severity        CNN only (softmax)    QWK 0.2488                                                                                                                           
 severity          CNN only (CORAL)    QWK 0.0000                                                                                                                           
 severity          + tissue, concat    QWK 0.9530 circular: tissue input includes the label-generating variable; large QWK gain reflects that, not a learned clinical signal
 severity + tissue, gated (TGO-Net)    QWK 0.8911                                                                                                                           
 severity   + tissue, gated + SMOTE    QWK 0.9111                                                                                      

In [5]:
# Cell 5 · Figure 1 — the four validation checks
if 'validation' in data:
    v = data['validation']
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
    fig.suptitle('Label validation: four independent checks',
                 fontsize=13, fontweight='bold', color=INK, x=0.02, ha='left')

    # (a) kappa values against the interpretive bands
    ax = axes[0]
    c = v.get('check1_expert', {})
    ks = [('derived\nvs A', c.get('kappa_derived_vs_A')),
          ('derived\nvs B', c.get('kappa_derived_vs_B')),
          ('A vs B\n(experts)', c.get('kappa_inter_rater'))]
    ks = [(n, k) for n, k in ks if k is not None]
    if ks:
        ax.bar([n for n,_ in ks], [k for _,k in ks],
               color=[CRIM if k < 0.21 else AMBER for _,k in ks], width=0.6)
        ax.axhline(0.21, color=MUTE, ls='--', lw=1)
        ax.text(0.02, 0.215, 'slight / fair boundary', fontsize=7, color=MUTE)
        ax.axhline(0, color=INK, lw=1)
        span = max(abs(min(k for _,k in ks)), max(k for _,k in ks))
        pad = span * 0.10
        for i, (_, k) in enumerate(ks):
            # place the label outside the bar on both signs, so a negative
            # value does not sit on top of the zero line
            ax.text(i, k + pad if k >= 0 else k - pad, f'{k:+.3f}',
                    ha='center', va='bottom' if k >= 0 else 'top',
                    fontsize=9, fontweight='bold', color=INK)
        ax.set_ylim(min(0, min(k for _,k in ks)) - pad*2.2,
                    max(k for _,k in ks) + pad*2.2)
        ax.set_ylabel("Cohen's kappa", fontsize=9, color=MUTE)
    ax.set_title('(a) expert agreement', fontsize=10, loc='left', color=INK)

    # (b) negative control
    ax = axes[1]
    c = v.get('check2_negctrl', {})
    pct = c.get('pct_severe')
    if pct is not None:
        ax.bar(['graded\nsevere','correct\nanswer'], [pct, 0],
               color=[CRIM, GREEN], width=0.55)
        ax.text(0, pct + 2, f'{pct:.1f}%', ha='center', fontsize=11,
                fontweight='bold', color=CRIM)
        ax.text(1, 2, '0%', ha='center', fontsize=11, fontweight='bold',
                color=GREEN)
        ax.set_ylim(0, 105); ax.set_ylabel('% of healthy skin', fontsize=9,
                                           color=MUTE)
    ax.set_title('(b) negative control', fontsize=10, loc='left', color=INK)

    # (c) instability
    ax = axes[2]
    c = v.get('check3_stability', {})
    pi = c.get('pct_inconsistent')
    if pi is not None:
        ax.pie([pi, 100-pi], labels=[f'{pi:.1f}%\ninconsistent','consistent'],
               colors=[CRIM, '#DCE6EA'], startangle=90,
               textprops={'fontsize':8.5})
    ax.set_title('(c) stability under augmentation', fontsize=10, loc='left',
                 color=INK)
    fig.tight_layout()
    save(fig, 'fig1_validation')
else:
    print('  FLAGGED: no validation data for figure 1')

  wrote figures/fig1_validation.png and .pdf


In [6]:
# Cell 6 · Figure 2 — control comparison, the paper's core claim
if 'severity' in data and 'infection' in data:
    cnn = [c for c in data['severity']['configs'] if c['name'].startswith('CNN only')]
    sev = max(cnn, key=lambda c: c['qwk']) if cnn else None
    inf = max(data['infection']['configs'], key=lambda c: c['auroc'])

    fig, ax = plt.subplots(figsize=(7.2, 3.6))
    names = ['derived colour labels\n(severity, QWK)',
             'expert clinician labels\n(infection, AUROC)']
    vals = [sev['qwk'] if sev else 0, inf['auroc']]
    bars = ax.bar(names, vals, color=[CRIM, TEAL], width=0.5)
    for b, v_ in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, v_+0.02, f'{v_:.3f}',
                ha='center', fontsize=12, fontweight='bold', color=INK)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('score from pixels alone', fontsize=9, color=MUTE)
    ax.set_title('Identical pipeline. Only the label source differs.',
                 fontsize=12, fontweight='bold', color=INK, loc='left')
    ax.text(0, -0.22, 'Same architectures, preprocessing, grouped folds and code.\n'
            'The gap isolates label provenance, not model capacity.',
            transform=ax.transAxes, fontsize=8.5, color=MUTE, va='top')
    fig.tight_layout()
    save(fig, 'fig2_control')
else:
    print('  FLAGGED: need both severity and infection results for figure 2')

  wrote figures/fig2_control.png and .pdf


In [7]:
# Cell 7 · Figure 3 — the gate never settled
if 'infection' in data:
    tgo = next((c for c in data['infection']['configs']
                if 'gamma_per_fold' in c), None)
    if tgo:
        g = np.array(tgo['gamma_per_fold'], dtype=float)
        fig, ax = plt.subplots(figsize=(6.6, 3.4))
        cols = [CRIM if x < 0 else TEAL for x in g]
        bars = ax.bar(range(1, len(g)+1), g, color=cols, width=0.55)
        pad = (g.max() - min(0, g.min())) * 0.06
        for b, v_ in zip(bars, g):
            ax.text(b.get_x()+b.get_width()/2, v_ + pad if v_ >= 0 else v_ - pad,
                    f'{v_:+.3f}', ha='center',
                    va='bottom' if v_ >= 0 else 'top',
                    fontsize=9, fontweight='bold', color=INK)
        ax.set_ylim(min(0, g.min()) - pad*3, g.max() + pad*3)
        ax.axhline(0, color=INK, lw=1.2)
        ax.axhline(g.mean(), color=MUTE, ls='--', lw=1)
        # annotate inside the axes; a label past the last bar gets clipped
        # put the mean label wherever the bars are shortest, so it never
        # lands on top of one
        left_room = g[:2].max() < g[-2:].max()
        ax.text(0.02 if left_room else 0.98, g.mean(),
                f'mean {g.mean():+.3f}',
                transform=ax.get_yaxis_transform(), fontsize=8,
                color=MUTE, va='bottom',
                ha='left' if left_room else 'right',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                          edgecolor='none', alpha=0.85))
        ax.set_xlabel('fold', fontsize=9, color=MUTE)
        ax.set_ylabel(r'learned gate scalar $\gamma$', fontsize=9, color=MUTE)
        ax.set_title('The gate did not converge on a use of tissue',
                     fontsize=12, fontweight='bold', color=INK, loc='left')
        sd = g.std()
        p = data.get('stats', {}).get('gate_test', {}).get('p_value')
        sub = (rf'$\gamma$ starts at 0 and is free to grow.  '
               rf'sd {sd:.3f} vs mean {abs(g.mean()):.3f}')
        if p is not None: sub += f',  p = {p:.3f} vs zero'
        ax.text(0, -0.24, sub, transform=ax.transAxes, fontsize=8.5,
                color=CRIM, va='top')
        fig.tight_layout()
        save(fig, 'fig3_gate')
    else:
        print('  FLAGGED: no gate values recorded')

  wrote figures/fig3_gate.png and .pdf


In [8]:
# Cell 8 · Figure 4 — class distribution and the mild problem
if 'labels' in data:
    df = data['labels']
    counts = df.severity.value_counts().reindex(['mild','moderate','severe'])
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))

    ax = axes[0]
    bars = ax.bar(counts.index, counts.values,
                  color=[GREEN, AMBER, CRIM], width=0.55)
    for b, v_ in zip(bars, counts.values):
        ax.text(b.get_x()+b.get_width()/2, v_ + max(counts)*0.02,
                f'{int(v_):,}', ha='center', fontsize=10,
                fontweight='bold', color=INK)
    ax.set_ylabel('photographs', fontsize=9, color=MUTE)
    ax.set_title('(a) consolidated class distribution', fontsize=10,
                 loc='left', color=INK)

    ax = axes[1]
    per_fold = counts['mild'] / 5
    ax.bar(['per fold'], [per_fold], color=GREEN, width=0.4)
    ax.text(0, per_fold + 0.3, f'{per_fold:.1f}', ha='center',
            fontsize=12, fontweight='bold', color=INK)
    ax.axhline(15, color=CRIM, ls='--', lw=1.2)
    ax.text(0.42, 15.5, 'minimum for a usable\nper-class estimate',
            fontsize=8, color=CRIM)
    ax.set_ylim(0, 22)
    ax.set_ylabel('mild photographs', fontsize=9, color=MUTE)
    ax.set_title('(b) why mild is unmeasurable', fontsize=10, loc='left',
                 color=INK)
    fig.suptitle('Class distribution after consolidation', fontsize=12,
                 fontweight='bold', color=INK, x=0.02, ha='left')
    fig.tight_layout()
    save(fig, 'fig4_classes')
else:
    print('  FLAGGED: labels_consolidated.csv not found')

  wrote figures/fig4_classes.png and .pdf


In [9]:
# Cell 9 · manifest — what exists, what is flagged
manifest = dict(
    figures=sorted(p.name for p in FIGS.glob('*.png')),
    tables=sorted(p.name for p in TABLES.glob('*.csv')),
    missing_sources=missing)
with open(OUT / 'figure_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print('=' * 58)
print('STAGE 10 COMPLETE')
print('=' * 58)
print(f'  figures ({len(manifest["figures"])}):')
for n in manifest['figures']: print(f'    {n}')
print(f'  tables ({len(manifest["tables"])}):')
for n in manifest['tables']: print(f'    {n}')
if missing:
    print(f'\n  FLAGGED missing sources: {missing}')
    print('  Figures depending on them were skipped, not faked.')
print(f'\n  wrote {(OUT / "figure_manifest.json").resolve()}')
print('\n  Every value above was read from a stored result file.')
print('  No number in the paper should be typed by hand.')

STAGE 10 COMPLETE
  figures (4):
    fig1_validation.png
    fig2_control.png
    fig3_gate.png
    fig4_classes.png
  tables (3):
    table1_validation.csv
    table2_control.csv
    table3_ablation.csv

  wrote /Users/mustaphanasser/Documents/computer_application/DFU_project/notebook2_outputs/figure_manifest.json

  Every value above was read from a stored result file.
  No number in the paper should be typed by hand.
